In [3]:
# Phase 1: Import Required Libraries

# Hugging Face Dataset Library
from datasets import load_dataset

# Hugging Face Pipeline
from transformers import pipeline

# Data Manipulation
import pandas as pd

# Progress Bar
from tqdm import tqdm

# Evaluation Metrics (used in later branches)
from sklearn.metrics import accuracy_score, f1_score

# PyTorch
import torch

In [4]:
# Phase 2: Check Device

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU")

Using CPU


In [ ]:
# Phase 3: Load IMDb Dataset

dataset = load_dataset("imdb")

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

c:\Users\Hp\Desktop\AI-ML Internship\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hp\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
# Phase 4: Explore the Dataset

# Display dataset information
print(dataset)
# Display available splits
print("\nAvailable Splits:")
print(dataset.keys())
# Display the first sample from the test dataset
print("\nFirst Test Sample:")
print(dataset["test"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Available Splits:
dict_keys(['train', 'test', 'unsupervised'])

First Test Sample:
{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sc

In [25]:
# Phase 5: Prepare Evaluation Dataset

# Number of samples for evaluation
sample_size = 500

# Shuffle the test dataset for a balanced sample
test_dataset = dataset["test"].shuffle(seed=42).select(range(sample_size))

print(f"Evaluation Dataset Size: {len(test_dataset)}")

Evaluation Dataset Size: 500


In [26]:
# Phase 6: Display Sample Reviews

sample_df = pd.DataFrame(test_dataset[:5])
sample_df

,text,label
0,<br /><br />When I unsuspectedly rented A Thou...,1
1,This is the latest entry in the long series of...,1
2,This movie was so frustrating. Everything seem...,0
3,"I was truly and wonderfully surprised at ""O' B...",1
4,This movie spends most of its time preaching t...,0


In [27]:
# Phase 7: Load Default Sentiment Pipeline

classifier = pipeline(
    task="sentiment-analysis",
    device=device
)
print("Default sentiment pipeline loaded successfully!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Default sentiment pipeline loaded successfully!


In [28]:
# Phase 8: Generate Predictions Using Default Pipeline

# Convert dataset column to a Python list
reviews = list(test_dataset["text"])

# Generate predictions
results = classifier(
    reviews,
    truncation=True,
    batch_size=16
)

# Extract predicted labels
default_predictions = [result["label"] for result in results]

print(f"Total Predictions: {len(default_predictions)}")
print("First 10 Predictions:")
print(default_predictions[:10])

Total Predictions: 500
First 10 Predictions:
['POSITIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE']


In [29]:

# Phase 9: Convert Default Predictions to Numeric Labels

# Convert text labels to numeric labels
default_predicted_labels = [
    1 if prediction == "POSITIVE" else 0
    for prediction in default_predictions
]

# Store actual labels
true_labels = list(test_dataset["label"])

print("First 10 Numeric Predictions:")
print(default_predicted_labels[:10])

print("\nFirst 10 True Labels:")
print(true_labels[:10])

First 10 Numeric Predictions:
[1, 1, 0, 0, 0, 1, 1, 0, 0, 1]

First 10 True Labels:
[1, 1, 0, 1, 0, 1, 1, 0, 0, 1]


In [30]:
# Phase 10: Load Task-Specific Sentiment Model
task_classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

print("Task-specific sentiment model loaded successfully!")

Device set to use cpu


Task-specific sentiment model loaded successfully!


In [31]:
# Phase 11: Test Task-Specific Model
sample_review = "This movie was absolutely amazing. I loved every minute of it."

result = task_classifier(sample_review)

print(result)

[{'label': 'POSITIVE', 'score': 0.9998840093612671}]


In [32]:
# Phase 12: Generate Predictions Using Task-Specific Model
# Convert dataset column to a Python list
reviews = list(test_dataset["text"])

# Generate predictions
results = task_classifier(
    reviews,
    truncation=True,
    batch_size=16
)

# Extract predicted labels
task_predictions = [result["label"] for result in results]

print(f"Total Predictions: {len(task_predictions)}")
print("First 10 Predictions:")
print(task_predictions[:10])

Total Predictions: 500
First 10 Predictions:
['POSITIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE']


In [33]:
# Phase 13: Convert Task-Specific Predictions to Numeric Labels
task_predicted_labels = [
    1 if prediction == "POSITIVE" else 0
    for prediction in task_predictions
]

print("First 10 Numeric Predictions:")
print(task_predicted_labels[:10])

First 10 Numeric Predictions:
[1, 1, 0, 0, 0, 1, 1, 0, 0, 1]


In [34]:
# Phase 14: Evaluate Default Pipeline

# Calculate Accuracy
default_accuracy = accuracy_score(
    true_labels,
    default_predicted_labels
)

# Calculate F1-Score
default_f1 = f1_score(
    true_labels,
    default_predicted_labels
)

print("Default Pipeline Performance")
print(f"Accuracy : {default_accuracy:.4f}")
print(f"F1-Score : {default_f1:.4f}")

Default Pipeline Performance
Accuracy : 0.8900
F1-Score : 0.8852


In [35]:
# Phase 15: Evaluate Task-Specific Model

# Calculate Accuracy
task_accuracy = accuracy_score(
    true_labels,
    task_predicted_labels
)

# Calculate F1-Score
task_f1 = f1_score(
    true_labels,
    task_predicted_labels
)

print("Task-Specific Model Performance")
print(f"Accuracy : {task_accuracy:.4f}")
print(f"F1-Score : {task_f1:.4f}")

Task-Specific Model Performance
Accuracy : 0.8900
F1-Score : 0.8852


In [36]:
# Phase 16: Compare Model Performance
comparison_df = pd.DataFrame({
    "Model": [
        "Default Pipeline",
        "Task-Specific Model"
    ],
    "Accuracy": [
        default_accuracy,
        task_accuracy
    ],
    "F1-Score": [
        default_f1,
        task_f1
    ]
})

comparison_df

,Model,Accuracy,F1-Score
0,Default Pipeline,0.89,0.885177
1,Task-Specific Model,0.89,0.885177


In [38]:
# Phase 17: Identify Best Performing Model
if default_accuracy > task_accuracy:
    print("Best Performing Model: Default Pipeline")

elif task_accuracy > default_accuracy:
    print("Best Performing Model: Task-Specific Model")

else:
    print("Both models achieved identical performance.")

Both models achieved identical performance.


# Sentiment Classification using Hugging Face Pipeline

## Objective

The objective of this project is to perform sentiment classification on the IMDb movie reviews dataset using the Hugging Face `pipeline()` API.

The workflow includes:
- Loading and exploring the IMDb dataset.
- Running sentiment classification using the default Hugging Face pipeline.
- Loading an explicit task-specific sentiment model (`distilbert-base-uncased-finetuned-sst-2-english`) and generating predictions.
- Evaluating both approaches using **Accuracy** and **F1-score**.
- Comparing the performance of the two pipelines.

## Result

Both the default pipeline and the explicitly loaded task-specific model achieved identical Accuracy and F1-score on the evaluation dataset, indicating that the default pipeline uses the same pretrained model for sentiment analysis.